# 1795. Rearrange Products Table

## Problem Description

A product price table stores prices in separate columns, one per store. A missing price means the product is not sold at that store.

### Table Schema

**Products**

| Column | Type | Meaning / constraint |
| --- | --- | --- |
| product_id | int | unique product identifier (key) |
| store1 | int | price at store1 or null |
| store2 | int | price at store2 or null |
| store3 | int | price at store3 or null |

### Task

Reshape the wide table into a long table with one row per product-store combination that has a non-null price. Return product_id, store (the original column name, such as store1), and price. A zero price is a valid price and must not be excluded. Any row order is accepted.

## Example

### Input

**products**

| product_id | store1 | store2 | store3 |
| --- | --- | --- | --- |
| 1 | 10 | NULL | 0 |
| 2 | NULL | 20 | NULL |

### Expected Output

| product_id | store | price |
| --- | --- | --- |
| 1 | store1 | 10 |
| 1 | store3 | 0 |
| 2 | store2 | 20 |

### Explanation

Product 1 has prices 10 at store1 and 0 at store3; store2 is missing. Product 2 has price 20 at store2. Missing combinations are omitted.

## Practice

The next code cell constructs exactly these example DataFrames. Try the blank practice cell before reading the solutions.

## References

The problem is restated for self-contained practice; the examples are original. Community links are for further reading, and implementations below are independently written.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/1700-1799/1795.Rearrange%20Products%20Table/README_EN.md) · [LeetCode](https://leetcode.com/problems/rearrange-products-table/)

[Community solution index (Silversaya, 2023)](https://leetcode.cn/discuss/post/3573106/zhe-shi-pandas30tian-ji-hua-de-ti-jie-qu-v0wv/) · [LeetCode discussions](https://leetcode.com/problems/rearrange-products-table/solutions/) · [Individual community solution](https://leetcode.cn/problems/rearrange-products-table/solutions/2373764/1795-mei-ge-chan-pin-zai-bu-tong-shang-d-ayll/)

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal
from decimal import Decimal, ROUND_HALF_UP

def percent(n,d):
    return float((Decimal(int(n))*100/Decimal(int(d))).quantize(Decimal('0.01'),rounding=ROUND_HALF_UP)) if d else None

products = pd.DataFrame([[1,10,None,0],[2,None,20,None]], columns=["product_id","store1","store2","store3"])
expected = pd.DataFrame([[1,"store1",10],[1,"store3",0],[2,"store2",20]], columns=["product_id","store","price"])
display(products)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Melt
wide to long 最直接。

In [ ]:
def solution_1(products):
    return products.melt(id_vars='product_id',var_name='store',value_name='price').dropna(subset=['price'])

result_1 = solution_1(products.copy())
display(result_1)

### Solution 2 — Stack
欄位變成 index level。

In [ ]:
def solution_2(products):
    return products.set_index('product_id').rename_axis(columns='store').stack().dropna().rename('price').reset_index()

result_2 = solution_2(products.copy())
display(result_2)

### Solution 3 — Concat
每個 store 可有不同轉換規則。

In [ ]:
def solution_3(products):
    parts = [products[['product_id',s]].rename(columns={s:'price'}).assign(store=s) for s in ['store1','store2','store3']]
    return pd.concat(parts,ignore_index=True).dropna(subset=['price'])[['product_id','store','price']]

result_3 = solution_3(products.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare readability, intermediate data size and handling of duplicate values, missing matches and empty groups.